# 03 — Parsing Bronze IoT Hub → DataFrame Silver

**Objectif** : lire un fichier NDJSON simulé (IoT Hub),
décoder chaque message, extraire les tags et produire
un DataFrame prêt pour l'écriture Silver  
**Input** : `data/ndjson_sim/*.json`  
**Output** : DataFrame pandas avec le schéma Silver  
**Hypothèse** : tous les tags sont présents, qualité ignorée pour l'instant

In [13]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [14]:
import os
import json
import base64
import hashlib
import re
from pathlib import Path
from datetime import timezone

import pandas as pd
from dotenv import load_dotenv

load_dotenv()
print("OK")

OK


In [15]:
NDJSON_DIR = Path("../data/ndjson_sim")

# Prendre le premier fichier disponible
files = sorted(NDJSON_DIR.glob("**/*.json"))
print(f"Fichiers disponibles : {len(files)}")
for f in files[:5]:
    print(f"  {f.name}")

# Sélectionner le fichier à parser
TARGET_FILE = files[2]
print(f"\nFichier sélectionné : {TARGET_FILE.name}")

with open(TARGET_FILE, "r", encoding="utf-8") as f:
    lines = [l.strip() for l in f if l.strip()]

print(f"Messages dans ce fichier : {len(lines)}")

Fichiers disponibles : 180
  04_32_0.json
  04_33_0.json
  04_34_0.json
  04_35_0.json
  04_36_0.json

Fichier sélectionné : 04_34_0.json
Messages dans ce fichier : 9


In [16]:
# Regex pour extraire les indices [i][j] des tags matriciels
RE_MATRIX = re.compile(r"\[(\d+)\]\[(\d+)\]$")


def decode_message(raw_line: str) -> dict:
    """Décode une ligne NDJSON IoT Hub → dict body + metadata"""
    envelope = json.loads(raw_line)
    body     = json.loads(base64.b64decode(envelope["Body"]).decode("utf-8"))
    return {
        "enqueued_at":  envelope["EnqueuedTimeUtc"],
        "device_id":    envelope["SystemProperties"]["connectionDeviceId"],
        "machine_id":   body["process_serial_number"],
        "box":          body["box"],
        "values":       {
            v["id"].split(".")[-1]: v["value"]
            for v in body["values"]
        },
        "timestamps": {
            v["id"].split(".")[-1]: v["timestamp"]
            for v in body["values"]
        }
    }


def extract_matrix(values: dict, tag_prefix: str) -> list[list[int]]:
    """
    Extrait une matrice 15x10 depuis les tags values
    tag_prefix : ex "final_candled_eggs" ou "laser1_light_transmitted_eggs"
    Retourne une liste de 15 listes de 10 entiers
    """
    matrix = [[0] * 10 for _ in range(15)]
    for tag_name, val in values.items():
        if not tag_name.startswith(tag_prefix):
            continue
        m = RE_MATRIX.search(tag_name)
        if not m:
            continue
        row = int(m.group(1)) - 1  # 0-based
        col = int(m.group(2)) - 1  # 0-based
        if 0 <= row < 15 and 0 <= col < 10:
            matrix[row][col] = int(val) if val is not None else 0
    return matrix


def matrix_to_compact(matrix: list[list[int]]) -> str:
    """Convertit la matrice 15x10 en chaîne compacte de 150 chars"""
    return "".join(str(cell) for row in matrix for cell in row)


def compute_counts(matrix: list[list[int]]) -> dict:
    """
    Recompute les comptages depuis la matrice
    0=manquant, 1=fertile, 2=mort précoce, 3=clair, 4=mort tardif
    Règle d'or : jamais copier les comptages depuis les tags Bronze
    """
    flat = [cell for row in matrix for cell in row]
    return {
        "n_total":      len(flat),
        "n_fertile":    flat.count(1),
        "n_clear":      flat.count(3),
        "n_early_dead": flat.count(2),
        "n_late_dead":  flat.count(4),
        "n_missing":    flat.count(0),
    }


def compute_tray_id(machine_id: str, candled_at_utc: str) -> str:
    """tray_id = SHA-256(machine_id | candled_at_utc) — déterministe"""
    raw = f"{machine_id}|{candled_at_utc}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


print("Fonctions de parsing définies.")

Fonctions de parsing définies.


In [17]:
def parse_file(filepath: Path) -> pd.DataFrame:
    """
    Parse un fichier NDJSON IoT Hub complet
    Retourne un DataFrame avec le schéma Silver
    """
    rows = []

    with open(filepath, "r", encoding="utf-8") as f:
        lines = [l.strip() for l in f if l.strip()]

    for line in lines:
        msg = decode_message(line)

        # Ignorer les messages qui ne sont pas des plateaux
        pmaf_trig = msg["values"].get("pmaf_trig", "")
        if pmaf_trig != "Tray":
            print(f"  [SKIP] pmaf_trig={pmaf_trig}")
            continue

        v = msg["values"]

        # Timestamp d'ancrage : depuis le tag candled_at
        candled_at_str = msg["timestamps"].get("pmaf_trig", msg["enqueued_at"])
        candled_at = pd.to_datetime(candled_at_str, utc=True)

        # Matrice et comptages (recomputés — jamais copiés depuis les tags)
        matrix     = extract_matrix(v, "final_candled_eggs")
        counts     = compute_counts(matrix)
        compact    = matrix_to_compact(matrix)

        # Lumière transmise — tableau plat 150 valeurs
        light_mat  = extract_matrix(v, "laser1_light_transmitted_eggs")
        light_flat = [cell for row in light_mat for cell in row]

        tray_id = compute_tray_id(msg["machine_id"], candled_at_str)

        row = {
            # Identifiants
            "tray_id":                  tray_id,
            "machine_id":               msg["machine_id"],
            # Timestamps
            "candled_at":               candled_at,
            "candled_date":             candled_at.date(),
            # Dimensions flock / trolley
            "flock":                    str(v.get("flock_number", "")),
            "trolley":                  str(v.get("trolley_name", "")),
            "tray_seq":                 int(v.get("setter_tray_number_flock", 0)),
            "flock_name":               str(v.get("flock_name", "")),
            "trolley_name":             str(v.get("trolley_name", "")),
            "caliber":                  str(v.get("caliber", "")),
            "setter_tray_number_flock": int(v.get("setter_tray_number_flock", 0)),
            # Comptages recomputés depuis la matrice
            **counts,
            "is_count_consistent":      counts["n_total"] == 150,
            # Données brutes
            "matrix_compact":           compact,
            "light_flat":               light_flat,
            # Alarmes
            "alarm_emergency_stop":     int(v.get("alarm_emergency_stop", 0)),
            "alarm_air_pressure_fault": int(v.get("alarm_air_pressure_fault", 0)),
            # Traçabilité
            "processed_at":             pd.Timestamp.utcnow(),
            "bronze_path":              str(filepath.name),
            "year":  candled_at.year,
            "month": candled_at.month,
            "day":   candled_at.day,
        }
        rows.append(row)

    return pd.DataFrame(rows)


df_parsed = parse_file(TARGET_FILE)
print(f"Plateaux parsés : {len(df_parsed)}")
df_parsed.head(3)

Plateaux parsés : 9


,tray_id,machine_id,candled_at,candled_date,flock,trolley,tray_seq,flock_name,trolley_name,caliber,...,is_count_consistent,matrix_compact,light_flat,alarm_emergency_stop,alarm_air_pressure_fault,processed_at,bronze_path,year,month,day
0,a101fe12710739387ed4ec0e96fe07eae93ed4895197db...,PMAF-C012501,2026-05-15 04:34:06.250856+00:00,2026-05-15,1,20713024,14,43e6ab99aeea0b2d6ca,20713024,1,...,True,1113111111111111311111131111311113111111131111...,"[172554, 196180, 135927, 0, 160919, 160811, 14...",0,0,2026-06-24 13:48:46.958728+00:00,04_34_0.json,2026,5,15
1,ead7f4368bfcf0033707fc862039755e549b6a41770932...,PMAF-C012501,2026-05-15 04:34:12.246406+00:00,2026-05-15,1,20713024,15,43e6ab99aeea0b2d6ca,20713024,1,...,True,1111111111111112111111121111111113113111111111...,"[159918, 185087, 185500, 160492, 148251, 14817...",0,0,2026-06-24 13:48:46.959761+00:00,04_34_0.json,2026,5,15
2,ed629a35946fe938854101d6ae432201511fe081c85663...,PMAF-C012501,2026-05-15 04:34:18.744640+00:00,2026-05-15,1,20713024,16,43e6ab99aeea0b2d6ca,20713024,1,...,True,1111111111111123111111111131111311111111111111...,"[159991, 185577, 185489, 37500, 186016, 184573...",0,0,2026-06-24 13:48:46.960685+00:00,04_34_0.json,2026,5,15


In [19]:
# Vérifier que le schéma produit correspond au schéma Silver attendu
SILVER_COLS = [
    "tray_id", "machine_id", "candled_at", "candled_date",
    "flock", "trolley", "tray_seq", "flock_name", "trolley_name",
    "caliber", "setter_tray_number_flock",
    "n_total", "n_fertile", "n_clear", "n_early_dead",
    "n_late_dead", "n_missing", "is_count_consistent",
    "matrix_compact", "light_flat",
    "alarm_emergency_stop", "alarm_air_pressure_fault",
    "processed_at", "bronze_path", "year", "month", "day"
]

missing = [c for c in SILVER_COLS if c not in df_parsed.columns]
extra   = [c for c in df_parsed.columns if c not in SILVER_COLS]

print(f"Colonnes attendues : {len(SILVER_COLS)}")
print(f"Colonnes produites : {len(df_parsed.columns)}")
print(f"Manquantes : {missing if missing else 'aucune '}")
print(f"En trop    : {extra   if extra   else 'aucune '}")

Colonnes attendues : 27
Colonnes produites : 27
Manquantes : aucune 
En trop    : aucune 


In [20]:
# Vérifier la règle d'or : comptages recomputés != tags Bronze
# On compare n_fertile avec #_living_embryo_flock (tag Bronze)
# Ils doivent être identiques car on a simulé depuis Silver
# En prod, toute divergence serait un bug

print("Validation des comptages (règle d'or) :")
print(f"  is_count_consistent=True  : {df_parsed['is_count_consistent'].sum()}")
print(f"  is_count_consistent=False : {(~df_parsed['is_count_consistent']).sum()}")
print()

# Distribution des classes
for col in ["n_total", "n_fertile", "n_clear", "n_early_dead", "n_late_dead", "n_missing"]:
    print(f"  {col:<25} min={df_parsed[col].min():>4}  "
          f"moy={df_parsed[col].mean():>6.1f}  "
          f"max={df_parsed[col].max():>4}")

Validation des comptages (règle d'or) :
  is_count_consistent=True  : 9
  is_count_consistent=False : 0

  n_total                   min= 150  moy= 150.0  max= 150
  n_fertile                 min= 129  moy= 135.9  max= 142
  n_clear                   min=   6  moy=  10.3  max=  17
  n_early_dead              min=   0  moy=   1.2  max=   3
  n_late_dead               min=   0  moy=   2.6  max=   7
  n_missing                 min=   0  moy=   0.0  max=   0


In [21]:
# Parser l'ensemble des fichiers simulés
all_dfs = []
for filepath in sorted(NDJSON_DIR.glob("**/*.json")):
    df_batch = parse_file(filepath)
    all_dfs.append(df_batch)

df_all = pd.concat(all_dfs, ignore_index=True)

print(f"Fichiers parsés   : {len(all_dfs)}")
print(f"Total plateaux    : {len(df_all)}")
print(f"Période           : {df_all['candled_at'].min()} → {df_all['candled_at'].max()}")
print(f"Doublons tray_id  : {df_all['tray_id'].duplicated().sum()}")

Fichiers parsés   : 180
Total plateaux    : 1344
Période           : 2026-05-15 04:32:48.638548+00:00 → 2026-05-15 07:57:28.697491+00:00
Doublons tray_id  : 0


In [22]:
df_all["flock"] = df_all["flock"].astype(int)

(
    df_all
    .groupby("flock")[["n_total", "n_fertile", "n_clear", "n_early_dead", "n_late_dead"]]
    .sum()
    .sort_index()
    .head(10)
)

,n_total,n_fertile,n_clear,n_early_dead,n_late_dead
flock,,,,,
1,4800,4402,311,29,58
2,4800,4403,307,34,56
3,4800,4375,334,40,51
4,4800,4392,299,41,68
5,4350,3996,284,27,43
6,6450,5903,435,48,64
7,4800,3974,679,38,109
8,4950,4043,775,54,78
9,4800,3923,708,47,122


In [ ]:
# Exportation PKL

import pickle
from pathlib import Path

output_path = Path("../data") / "df_all.pkl"
df_all.to_pickle(output_path)
print(f"df_all exporté : {output_path}")
print(f"Lignes : {len(df_all)}")

df_all exporté : ..\data\df_all.pkl
Lignes : 1344


: 